**1. Preparando o Ambiente e criando o banco de dados**

In [19]:
import sqlite3
import pandas as pd

# 1. Ler o seu arquivo CSV
df = pd.read_csv('/content/personal_transactions_enriquecido.csv')

# 2. Criar a conexão com o banco de dados (isso cria o arquivo .db automaticamente)
conn = sqlite3.connect('projeto_hackathon.db') #banco de dados final

# 3. Converter a coluna de Data para o formato que o SQL entende (YYYY-MM-DD)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')

# 4. Criar a tabela e inserir os dados de uma vez só
# O comando 'to_sql' faz todo o trabalho de "ligar" o CSV ao Banco.
df.to_sql('transacoes', conn, if_exists='replace', index=False)

# 5. Testar se funcionou (Contar as linhas)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM transacoes")
total = cursor.fetchone()[0]

print(f"Banco de dados criado com sucesso!")
print(f"Tabela 'transacoes' alimentada com {total} linhas.")

conn.close()

Banco de dados criado com sucesso!
Tabela 'transacoes' alimentada com 19046 linhas.


*Para garantir a robustez da análise de saúde financeira, os dados brutos em formato CSV foram processados via linguagem Python e armazenados em um banco de dados relacional SQLite. Este processo incluiu a normalização de tipos de dados e a padronização temporal, permitindo a execução de queries SQL complexas para extração de insights sobre padrões de consumo e margem de poupança mensal.*

1. Query Variação de Despesas -
Análise da saúde financeira ao longo do tempo. Esta query calcula o total de ganhos (créditos) e o total de gastos (débitos) para mostrar o saldo final.
**Pergunta: O usuário fecha o mês no azul ou no vermelho?**

In [34]:
import sqlite3
import pandas as pd

query_saldo_mensal = """
SELECT
    strftime('%Y-%m', Date) as Mes,
    ROUND(SUM(CASE WHEN "Transaction Type" = 'credit' THEN Amount ELSE 0 END), 2) as Entradas,
    ROUND(SUM(CASE WHEN "Transaction Type" = 'debit' THEN Amount ELSE 0 END), 2) as Saidas,
    ROUND(SUM(CASE WHEN "Transaction Type" = 'credit' THEN Amount ELSE 0 END) -
          SUM(CASE WHEN "Transaction Type" = 'debit' THEN Amount ELSE 0 END), 2) as Saldo_Mensal
FROM transacoes
GROUP BY Mes
ORDER BY Mes;
"""

# Reabrir a conexão com o banco de dados
conn = sqlite3.connect('projeto_hackathon.db')

# Executar a query e carregar os resultados em um DataFrame
df_saldo_mensal = pd.read_sql_query(query_saldo_mensal, conn);

# Fechar a conexão
conn.close()

# Exibir os resultados
print(df_saldo_mensal);

        Mes   Entradas     Saidas  Saldo_Mensal
0   2018-01    7162.89    2931.45       4231.44
1   2018-02    5220.75    3165.05       2055.70
2   2018-03    7321.50    3500.16       3821.34
3   2018-04    7166.88    6029.54       1137.34
4   2018-05    5091.55   11392.03      -6300.48
5   2018-06    6017.19    3665.88       2351.31
6   2018-07    4666.34    2968.98       1697.36
7   2018-08    7379.15    2396.18       4982.97
8   2018-09    5234.71    3286.99       1947.72
9   2018-10    5022.23    2848.35       2173.88
10  2018-11    6018.96    2963.65       3055.31
11  2018-12    5635.10    3427.99       2207.11
12  2019-01    4769.44    5187.31       -417.87
13  2019-02    4500.01    3163.40       1336.61
14  2019-03    7792.25    3241.51       4550.74
15  2019-04    5931.89    4829.55       1102.34
16  2019-05    5341.01    4673.51        667.50
17  2019-06    4514.35   11999.60      -7485.25
18  2019-07    5642.40    4148.06       1494.34
19  2019-08    7304.10    4266.17       

*O banco de dados revela uma mudança de perfil financeiro ou de escala de movimentação a partir de 2023, exigindo uma nova calibração das metas de economia.*

2. Query Ranking dos Gastos (Top 10 Categorias)

**Pergunta: Quais são os 10 maiores grupos de despesa?**

In [36]:
import sqlite3
import pandas as pd

query_top_gastos = """
SELECT
    Category as Categoria,
    ROUND(SUM(Amount), 2) as Gasto_Total
FROM transacoes
WHERE "Transaction Type" = 'debit'
GROUP BY Category
ORDER BY Gasto_Total DESC
LIMIT 10;
"""

# Reabrir a conexão com o banco de dados
conn = sqlite3.connect('projeto_hackathon.db')

# Executar a query e carregar os resultados em um DataFrame
df_top_gastos = pd.read_sql_query(query_top_gastos, conn)

# Fechar a conexão
conn.close()

# Exibir os resultados
print(df_top_gastos)

             Categoria  Gasto_Total
0                 Rent   1591300.84
1  Credit Card Payment    425432.19
2            Groceries    340873.59
3            Insurance    264164.37
4         Loan Payment    217126.37
5     Savings Transfer    215839.74
6            Utilities    193010.95
7           Gas & Fuel    161376.34
8               Travel    132858.62
9                Taxes    132280.45


*O aluguel (Rent) é disparado a maior conta. Insight: "Como o custo fixo de habitação é muito alto, a saúde financeira do usuário depende de um controle rigoroso nos gastos variáveis (como Groceries e Gas), que são as próximas categorias no ranking.*

3. Query Análise de Gastos Recorrentes (Contas Fixas)

 Gastos que aparecem em quase todos os meses geralmente são assinaturas ou contas fixas.

**Pergunta: Quais despesas se repetem mensalmente?**


In [38]:
import sqlite3
import pandas as pd

query_gastos_recorrentes = """
SELECT
    Description as Descricao,
    COUNT(DISTINCT strftime('%m', Date)) as Meses_Ativos,
    ROUND(AVG(Amount), 2) as Valor_Medio
FROM transacoes
WHERE "Transaction Type" = 'debit'
GROUP BY Description
HAVING Meses_Ativos >= 10
ORDER BY Meses_Ativos DESC;
"""

# Reabrir a conexão com o banco de dados
conn = sqlite3.connect('projeto_hackathon.db')

# Executar a query e carregar os resultados em um DataFrame
df_gastos_recorrentes = pd.read_sql_query(query_gastos_recorrentes, conn);

# Fechar a conexão
conn.close()

# Exibir os resultados
print(df_gastos_recorrentes);

                    Descricao  Meses_Ativos  Valor_Medio
0                  State Farm            12        75.00
1                     Spotify            12        10.69
2               Power Company            12        60.00
3               Phone Company            12        80.02
4                     Netflix            12        12.30
5            Mortgage Payment            12      1178.79
6   Internet Service Provider            12        74.80
7               Grocery Store            12        26.84
8                 Gas Company            12        37.19
9         Credit Card Payment            12       465.37
10         City Water Charges            12        35.00
11                     Amazon            12        33.39
12             Smith and Sons            11       273.89
13               Williams LLC            10       216.72
14                  Starbucks            10         3.91
15                  Smith PLC            10       270.42
16                  Smith Inc  

*A economia por 'assinaturas esquecidas' é uma oportunidade imediata de melhoria no saldo mensal.*

4. Query Perfil de Pagamento

Esta query mostra se o usuário usa mais cartão de crédito ou conta corrente, o que ajuda a entender o comportamento de crédito.

**Pergunta: Qual o volume de gastos por cada conta cadastrada?**

In [40]:
import sqlite3
import pandas as pd

query_perfil_pagamento = """
SELECT
    "Account Name" as Conta,
    COUNT(*) as Numero_de_Transacoes,
    ROUND(SUM(Amount), 2) as Total_Gasto
FROM transacoes
WHERE "Transaction Type" = 'debit'
GROUP BY "Account Name"
ORDER BY Total_Gasto DESC;
"""

# Reabrir a conexão com o banco de dados
conn = sqlite3.connect('projeto_hackathon.db')

# Executar a query e carregar os resultados em um DataFrame
df_perfil_pagamento = pd.read_sql_query(query_perfil_pagamento, conn);

# Fechar a conexão
conn.close()

# Exibir os resultados
print(df_perfil_pagamento);

           Conta  Numero_de_Transacoes  Total_Gasto
0       checking                 10008   3698716.06
1    credit card                  7032    750035.40
2       Checking                   218     82498.14
3  Platinum Card                   324      8996.31
4    Silver Card                   146      4589.33


*A análise do Perfil de Pagamento revelou que a conta corrente é o principal veículo de despesa, concentrando mais de 10 mil transações. Isso demonstra um fluxo de caixa de alta rotatividade. O uso do cartão de crédito, embora secundário em volume de transações, representa uma fatia importante do valor total gasto, o que exige atenção à capacidade de endividamento futura do usuário.*